# Fisher forecast

The parameter constraints of a Euclid-like spectroscopic galaxy survey.

A Fisher forecast needs the derivatives of the observable with respect to every parameter. Here the observable is the galaxy power spectrum in four redshift bins, with the Kaiser effect and the Alcock-Paczynski distortion of a survey analysed in a reference cosmology:

$$
P_\mathrm{obs}(k_\mathrm{ref}, \mu_\mathrm{ref}) = \frac{(b + f \mu^2)^2 P(k)}{q_\perp^2 q_\parallel}, \qquad
q_\perp = \frac{D_A}{D_{A,\mathrm{ref}}}, \qquad
q_\parallel = \frac{H_\mathrm{ref}}{H} .
$$

The Fisher matrix sums the bins of volume $V$ and galaxy density $n$,

$$
F_{ij} = \sum_\mathrm{bins} \frac{V}{4 \pi^2} \int_0^1 \mathrm{d}\mu \int \mathrm{d}k \, k^2 \,
\partial_i \ln P_\mathrm{obs} \, \partial_j \ln P_\mathrm{obs}
\left(\frac{n P_\mathrm{obs}}{1 + n P_\mathrm{obs}}\right)^2 ,
$$

and the diagonal of its inverse gives the errors marginalised over the rest.

This minimal model leaves out the non-linear damping, redshift errors and shot-noise nuisance parameters of the Euclid forecast it follows.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from discoeb import LCDM, solve_eb

## The fiducial cosmology

Given by $\Omega_m$, $\Omega_b$ and $H_0$, the parameters the forecast is in. `from_density_parameters` turns them into the model's physical densities, in JAX, so the derivatives go through it.

In [2]:
h = 0.67
FIDUCIAL = dict(
    Omegam=0.32,
    Omegab=0.05,
    H0=100.0 * h,
    A_s=2.1064e-9,
    n_s=0.96,
    Tcmb=2.7255,
    YHe=0.248,
    Neff=3.046,
)

## The survey

Bins, biases, volumes and galaxy densities follow the Euclid forecast of [arXiv:1910.09273](https://arxiv.org/abs/1910.09273).

In [3]:
# The survey: 15000 square degrees in four redshift bins.
REDSHIFTS = np.array([1.0, 1.2, 1.4, 1.65])
BIASES = np.array([1.46, 1.61, 1.75, 1.90])
VOLUMES = np.array([7.94, 9.15, 10.05, 16.22]) * 1.0e9 / h**3  # Mpc^3
GALAXIES = np.array([1815.0, 1701.5, 1410.0, 940.97]) * 15000.0  # per unit redshift
DENSITIES = GALAXIES * np.array([0.2, 0.2, 0.2, 0.3]) / VOLUMES  # 1/Mpc^3

K = np.geomspace(2.0e-4, 0.5, 128)  # wave numbers of the solve, 1/Mpc
K_REF = np.geomspace(1.0e-3 * h, 0.25 * h, 96)  # those the survey uses
MU = np.linspace(0.0, 1.0, 33)

## The parameters

Four of the cosmology, $\ln 10^{10} A_s$ and one bias per bin. The logarithm keeps the Fisher matrix well scaled.

In [4]:
# The parameters: four of the cosmology, ln(10^10 A_s) and the bins' biases.
COSMOLOGICAL = ("Omegam", "Omegab", "H0", "n_s")
NAMES = (*COSMOLOGICAL, "ln10^10A_s", *(f"b(z={z})" for z in REDSHIFTS))
theta = jnp.asarray(
    [
        *(FIDUCIAL[key] for key in COSMOLOGICAL),
        np.log(1.0e10 * FIDUCIAL["A_s"]),
        *BIASES,
    ]
)

## Save points

`solve_eb` integrates in the scale factor and differentiates at fixed scale factor. It saves at the four bins' redshifts and today, and every derivative below is at fixed redshift.

In [5]:
# The bins' scale factors, ascending as the solve saves them, then today.
A_BINS = 1.0 / (1.0 + REDSHIFTS)
A_SAVE = np.concatenate((A_BINS[::-1], [1.0]))

## Linear theory in every bin

One `solve_eb` gives $P(k)$, the growth rate $f = -k (z + f_b v_b) / (a H \delta)$ and $a H$ at each bin's redshift. The conformal time `tau` is a state of the solve, and the comoving distance to a bin is its value today less its value there.

In [6]:
def linear_theory(cosmology):
    """``P(k)`` and the growth rate ``f(k)``, each ``(bin, k)``, the distance and ``a H``, each ``(bin,)``."""

    values = solve_eb(
        cosmology, K, A_SAVE, keep=("clxc", "clxb", "z", "vb", "adotoa", "tau")
    )
    tau0 = values[0, -1, 0, -1]  # the conformal time today
    # the bins, in the order of REDSHIFTS: each (k, bin)
    clxc, clxb, z, vb, adotoa, tau = values[0, :, :, -2::-1]

    k = K[:, None]
    f_b = cosmology.omega_b / (cosmology.omega_c + cosmology.omega_b)
    delta = (1.0 - f_b) * clxc + f_b * clxb
    growth_rate = -k * (z + f_b * vb) / (adotoa * delta)
    primordial = cosmology.A_s * (k / cosmology.k_p) ** (cosmology.n_s - 1.0)
    pk = 2.0 * jnp.pi**2 / k**3 * primordial * delta**2
    return pk.T, growth_rate.T, tau0 - tau[0], adotoa[0]

The reference distances and expansion rates, which the survey assumes:

In [7]:
_, _, distance_ref, aH_ref = linear_theory(LCDM.from_density_parameters(**FIDUCIAL))

## The observed spectrum

One bin's spectrum on the survey's $(k_\mathrm{ref}, \mu_\mathrm{ref})$ grid, at the true wave number and angle the reference ones map to.

In [8]:
def observed_power(pk, growth_rate, q_perp, q_par, bias):
    """One bin's ``P_obs(k_ref, mu_ref)`` in Mpc^3, ``(len(K_REF), len(MU))``."""

    stretch = jnp.sqrt(1.0 + MU**2 * ((q_perp / q_par) ** 2 - 1.0))
    k = K_REF[:, None] / q_perp * stretch  # the true wave number and angle
    mu = MU * q_perp / q_par / stretch
    pk = jnp.exp(jnp.interp(jnp.log(k), np.log(K), jnp.log(pk)))
    growth_rate = jnp.interp(jnp.log(k), np.log(K), growth_rate)
    return (bias + growth_rate * mu**2) ** 2 * pk / (q_perp**2 * q_par)

`log_power` goes from the parameters to $\ln P_\mathrm{obs}$ of every bin, with `jax.vmap` over the bins.

In [9]:
def log_power(theta):
    """``ln P_obs`` of every bin, ``(bin, len(K_REF), len(MU))``."""

    cosmology = LCDM.from_density_parameters(
        **{
            **FIDUCIAL,
            **dict(zip(COSMOLOGICAL, theta)),
            "A_s": 1.0e-10 * jnp.exp(theta[4]),
        }
    )
    pk, growth_rate, distance, aH = linear_theory(cosmology)
    power = jax.vmap(observed_power)(
        pk, growth_rate, distance / distance_ref, aH_ref / aH, theta[-len(BIASES) :]
    )
    return jnp.log(power)

## The derivatives

One `jax.jacfwd` gives the derivatives with respect to all nine parameters.

In [10]:
power = jnp.exp(log_power(theta))
derivatives = jax.jacfwd(log_power)(theta)  # (bin, k, mu, parameter)

## The Fisher matrix

The integrals over $\mu$ and $\ln k$ are trapezoid rules, with $k^2 \mathrm{d}k = k^3 \mathrm{d} \ln k$.

In [11]:
signal = DENSITIES[:, None, None] * power
weight = (signal / (1.0 + signal)) ** 2 * K_REF[:, None] ** 3  # k^2 dk = k^3 dln k
integrand = jnp.einsum("bkm,bkmi,bkmj->bkmij", weight, derivatives, derivatives)
over_mu = jnp.trapezoid(integrand, MU, axis=2)
per_bin = jnp.trapezoid(over_mu, np.log(K_REF), axis=1)  # (bin, parameter, parameter)
fisher = jnp.einsum("b,bij->ij", VOLUMES / (4.0 * jnp.pi**2), per_bin)

## The constraints

The marginalised errors, and for comparison the errors with every other parameter fixed.

In [12]:
marginalised = np.sqrt(np.diag(np.linalg.inv(fisher)))
conditional = 1.0 / np.sqrt(np.diag(fisher))
print(
    f"{'parameter':>10}  {'fiducial':>10}  {'1 sigma':>10}  {'relative':>9}  {'others fixed':>12}"
)
for name, value, sigma, fixed in zip(NAMES, theta, marginalised, conditional):
    print(
        f"{name:>10}  {value:10.4g}  {sigma:10.2e}  {sigma / value:9.2%}  {fixed:12.2e}"
    )

 parameter    fiducial     1 sigma   relative  others fixed
    Omegam        0.32    4.05e-03      1.27%      1.61e-04
    Omegab        0.05    8.71e-04      1.74%      6.53e-05
        H0          67    5.49e-01      0.82%      2.35e-02
       n_s        0.96    6.68e-03      0.70%      9.47e-04
ln10^10A_s       3.048    1.36e-02      0.45%      7.95e-04
  b(z=1.0)        1.46    6.61e-03      0.45%      1.34e-03
  b(z=1.2)        1.61    7.20e-03      0.45%      1.47e-03
  b(z=1.4)        1.75    7.79e-03      0.45%      1.70e-03
 b(z=1.65)         1.9    8.44e-03      0.44%      1.84e-03
